In [ ]:
# Cel: NAOCZNE, odpalalne porównanie naszej metody z kodem zespołu Homoli
# (znalezionym przez Maćka na credo-internal, 2026-08-19,
# źródła/kod/{pdf_values6.py,pdf_values8.py,dane/*}) - nie tylko moje
# twierdzenia z rozmowy, tylko konkretne liczby policzone tutaj, do
# samodzielnego sprawdzenia.
#
# Pytanie: dlaczego przy DOKŁADNIE tym samym przepisie (P=1675, d=5,
# m=4.0, dt=15, t0=14 lis 2013 07:00:00) artykuł podaje N+=218,N-=113,
# a u nas wychodzi N+=206,N-=128 (20260717.txt)?
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from scipy.stats import binom, norm

# Nasze dane/kod (jak we wszystkich poprzednich notebookach tego repo)
USGS_PATH = "../data/usgs_data/usgs_m4_2005_2025.csv"
MOSC_PATH = "../data/mosc_data.csv"

# Ich dane/kod, ściągnięte przez Maćka z credo-internal
THEIR_DIR = "../źródła/kod/dane/"

T0 = pd.Timestamp("2013-11-14 07:00:00")
P_DAYS = 1675
D_DAYS = 5
DT_DAYS = 15
M_THRESHOLD = 4.0

ARTICLE_RESULT = dict(Np=218, Nm=113, PCDF=4.1e-9)


In [ ]:
# NASZ kod (kopia cosmoseismic_stat z wcześniejszych notebooków, bez
# zmian) + wariant "detailed", który zwraca też tabelę bin-po-binie
# (żeby dało się porównać z ich zapisanym plikiem pdf_Mosc_...csv).
def load_earthquakes(min_mag=4.0):
    df = pd.read_csv(USGS_PATH, usecols=["time", "mag"])
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
    df = df[df["mag"] >= min_mag]
    return df.set_index("time")["mag"].sort_index()


def load_mosc(path=MOSC_PATH):
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.set_index("datetime").sort_index()["value"]


def cosmoseismic_stat_detailed(cr, eq, t0, P_days, d_days, m, dt_days):
    N = int(P_days // d_days)
    edges = pd.date_range(t0, periods=N + 1, freq=pd.Timedelta(days=d_days))
    eq_edges = edges + pd.Timedelta(days=dt_days)

    cr_cats = pd.cut(cr.index, edges, right=False)
    cr_binned = cr.groupby(cr_cats, observed=False).mean().reindex(cr_cats.categories)
    cr_vals = cr_binned.to_numpy()

    eq_in_range = eq[(eq.index >= eq_edges[0]) & (eq.index < eq_edges[-1])]
    eq_cats = pd.cut(eq_in_range.index, eq_edges, right=False)
    eq_binned = eq_in_range.groupby(eq_cats, observed=False).sum().reindex(eq_cats.categories, fill_value=0.0)
    sm_vals = eq_binned.to_numpy()

    nCR_i, nCR_im1 = cr_vals[1:], cr_vals[:-1]
    dCR = nCR_i - nCR_im1
    Sm = sm_vals[1:]

    med_Sm = np.nanmedian(Sm)
    med_dCR = np.nanmedian(np.abs(dCR))

    A = Sm / med_Sm - 1
    B = np.abs(dCR) / med_dCR - 1

    valid = (
        (A != 0) & (B != 0) &
        (nCR_i > 0) & (nCR_im1 > 0) &
        (Sm > 0) &
        ~np.isnan(A) & ~np.isnan(B)
    )

    bins = pd.DataFrame({
        "cr_bin_start": edges[1:-1],
        "nCR_i": nCR_i, "nCR_im1": nCR_im1, "dCR": dCR, "Sm": Sm,
        "A": A, "B": B, "valid": valid,
    })
    bins["C"] = np.sign(bins["A"] * bins["B"])
    bins = bins[bins["valid"]].reset_index(drop=True)

    Np, Nm = int((bins["C"] > 0).sum()), int((bins["C"] < 0).sum())
    n_total = Np + Nm
    ppdf = binom.pmf(Np, n_total, 0.5) if n_total else np.nan
    pcdf = binom.sf(Np - 1, n_total, 0.5) if n_total else np.nan
    sigma = norm.isf(pcdf) if n_total else np.nan

    return dict(N=N, N_valid=n_total, Np=Np, Nm=Nm, PPDF=ppdf, PCDF=pcdf, sigma=sigma), bins


eq_ours = load_earthquakes(min_mag=4.0)
cr_ours = load_mosc()
print(f"nasz EQ: {len(eq_ours)} zdarzeń, {eq_ours.index.min()} .. {eq_ours.index.max()}")
print(f"nasz CR (mosc_data.csv): {len(cr_ours)} pomiarów")


In [ ]:
# KROK 1: baseline - potwierdzamy tutaj (nie tylko z pamięci/dziennika),
# że nasz kod na naszych danych daje 206/128, tak jak w 20260717.txt.
baseline_stat, baseline_bins = cosmoseismic_stat_detailed(
    cr_ours, eq_ours, T0, P_DAYS, D_DAYS, M_THRESHOLD, DT_DAYS)
print("NASZ kod + NASZE dane:")
print(f"  N={baseline_stat['N']}, N_valid={baseline_stat['N_valid']}, "
      f"Np={baseline_stat['Np']}, Nm={baseline_stat['Nm']}, "
      f"PCDF={baseline_stat['PCDF']:.3e}, sigma={baseline_stat['sigma']:.3f}")
print(f"ARTYKUŁ: Np={ARTICLE_RESULT['Np']}, Nm={ARTICLE_RESULT['Nm']}, PCDF={ARTICLE_RESULT['PCDF']:.2e}")


In [ ]:
# KROK 2: porównanie SUROWYCH danych CR Moskwy - nasze vs ich dwa pliki
# (mosc_data1.csv, mosc_data2.csv). To bezpośredni test hipotezy "cicha
# rewizja danych NMDB od 2023" (20260717.txt/20260819.txt) - jeśli ich
# wartości są identyczne z naszymi, hipoteza upada.
def load_their_cr(filename):
    df = pd.read_csv(THEIR_DIR + filename)
    df["datetime"] = pd.to_datetime(df["datetime"], format="%Y-%m-%d %H:%M:%S")
    return df.set_index("datetime").sort_index()["value"]

cr_theirs_1 = load_their_cr("mosc_data1.csv")
cr_theirs_2 = load_their_cr("mosc_data2.csv")

merged = pd.DataFrame({"nasze": cr_ours}).join(
    pd.DataFrame({"ich_1": cr_theirs_1, "ich_2": cr_theirs_2}), how="inner")
merged["diff_1"] = merged["nasze"] - merged["ich_1"]
merged["diff_2"] = merged["nasze"] - merged["ich_2"]

print(f"Wspólnych znacznikow czasu: {len(merged)}")
print(f"mosc_data1.csv vs nasze: max|diff|={merged['diff_1'].abs().max():.3f}, "
      f"ile wartości ujemnych w mosc_data1: {(cr_theirs_1 <= 0).sum()}/{len(cr_theirs_1)} "
      f"({(cr_theirs_1 <= 0).mean()*100:.1f}%) -> prawdopodobnie wariant % (jak nasz units=1 z wczoraj)")
print(f"mosc_data2.csv vs nasze: max|diff|={merged['diff_2'].abs().max():.6f} "
      f"-> {'IDENTYCZNE' if merged['diff_2'].abs().max() < 1e-6 else 'RÓŻNE'}")
merged[["nasze", "ich_1", "ich_2", "diff_1", "diff_2"]].head(5)


In [ ]:
# KROK 2b: LOKALIZACJA rozbieżności mosc_data2.csv vs nasze mosc_data.csv
# (KROK 2 pokazał max|diff|=1905.72 na całym wspólnym zakresie - to
# ogromna wartość jak na surowe zliczenia CR ~100-300, więc prawdopodobnie
# lokalny problem (jak znany uszkodzony wiersz w oulu_5min_data.csv -
# patrz CLAUDE.md), nie systematyczne przesunięcie całego szeregu).
# Pełny wykres różnicy w czasie (CAŁY szereg, nie tylko wyciągnięte
# ekstremum - zgodnie z zasadą "pokazuj pełne dane" z rozmowy o wykresach
# dla Homoli) + tabela punktów, gdzie różnica przekracza próg szumu.
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(merged.index, merged["diff_2"], lw=0.5, color="steelblue")
ax.set_ylabel("różnica CR (nasze dane - archiwum)")
ax.set_xlabel("data")
ax.set_title("Różnica CR Moskwa: nasze dane vs mosc_data.csv z archiwum CREDO (wersja z 20.10.2023)")
ax.grid(ls=":", alpha=0.5)
fig.tight_layout()
fig.savefig("../results/mosc_data_versions_diff_plot.png", dpi=120)
plt.show()

DIFF_THRESHOLD = 1.0  # dużo większe niż typowa precyzja zapisu (~0.001-0.01)
diverge = merged[merged["diff_2"].abs() > DIFF_THRESHOLD]
print(f"Punktów z |diff_2| > {DIFF_THRESHOLD}: {len(diverge)} / {len(merged)} "
      f"({len(diverge) / len(merged) * 100:.3f}%)")
if len(diverge):
    print(f"Zakres dat rozbieżności: {diverge.index.min()} .. {diverge.index.max()}")
    print("\nNajwiększe rozbieżności:")
    print(diverge.reindex(diverge["diff_2"].abs().sort_values(ascending=False).index)
          [["nasze", "ich_2", "diff_2"]].head(15))


In [ ]:
# WAŻNE ZNALEZISKO: `pd.Grouper(freq="5D", origin="start")`, na którym
# opiera się CAŁA logika binowania w pdf_values6.py/pdf_values8.py, w
# pandas 3.0 (zainstalowana wersja - sprawdzone `pandas.__version__`)
# przestał działać dla częstotliwości wielodniowych - origin jest po
# cichu IGNOROWANY (RuntimeWarning: "The origin keyword does not take
# effect when resampling with a freq that is not Tick-like (h, m, s, ms,
# us, ns)" - w pandas 3.0 'D' (dzień) NIE jest już uznawany za "Tick-like"
# do tego celu). Sprawdzone eksperymentalnie: dotyczy to RÓWNIEŻ
# przekazania origin jako konkretnego Timestampu, nie tylko stringa
# "start" - więc nie da się tego obejść samym argumentem.
#
# Zamiast obniżać wersję pandas w całym współdzielonym środowisku (ryzyko:
# wpłynęłoby na WSZYSTKIE inne notebooki w tym repo, np. mc_parallel.py ma
# już własne zależności od zachowania konkretnej wersji pandas - patrz
# komentarz o Python 3.14/forkserver), odtwarzamy RĘCZNIE dokładnie to,
# co robił `origin="start"` przed pandas 3.0: biny zaczynające się
# dokładnie w pierwszym wierszu danych (po odfiltrowaniu), o stałej
# długości `freq_days` dni, przez pd.cut zamiast Grouper - to jest
# deterministyczne, działa identycznie w KAŻDEJ wersji pandas (sprawdzone
# ręcznie na przykładzie syntetycznym - wynik pokrywa się z oczekiwaną
# matematyką: pierwszy bin = średnia z pierwszych `freq_days*24/6`
# próbek 6h).
def bin_origin_start(df, key, value_col, freq_days, agg_funcs):
    origin = df[key].min()
    span_days = (df[key].max() - origin) / pd.Timedelta(days=1)
    n_bins = int(span_days // freq_days) + 1
    edges = pd.date_range(origin, periods=n_bins + 1, freq=pd.Timedelta(days=freq_days))
    cats = pd.cut(df[key], edges, right=False)
    grouped = df.groupby(cats, observed=False)[value_col].agg(agg_funcs)
    grouped = grouped.reindex(cats.cat.categories)
    grouped = grouped.reset_index(drop=True)
    grouped.insert(0, key, edges[:-1])
    return grouped


In [ ]:
# KROK 3: ICH kod - wierna kopia funkcji pdf() z pdf_values8.py (jedyna
# zmiana: nazwy plików CR/EQ jako parametry, zamiast wpisanych na sztywno
# "mosc_data.csv"/"eq_data.csv" - żeby dało się podstawić dowolny wariant
# danych bez nadpisywania plików źródłowych). CAŁA reszta logiki
# (filtrowanie, grupowanie, wzory) skopiowana 1:1, WŁĄCZNIE z tym, że ta
# wersja ma zakomentowane dodawanie sztucznego "zerowego" wiersza na
# początku (autor: "NIE WIEM CZY TRZEBA DODAC CZY NIE") - dokładnie tak
# jak w pliku źródłowym. Autor sam oznaczył ten plik jako
# "NIE DOKONCA DOBRE WYNIKI" (niedokończone/niepoprawne wyniki).
def their_pdf_v8(d, l, t, shift, shift2, station, cr_filename, eq_filename="eq_data.csv"):
    d = pd.to_datetime(d) + pd.Timedelta(days=shift2)
    d_end = d + pd.Timedelta(days=l * int(t[:-1]))

    eq_df = pd.read_csv(THEIR_DIR + eq_filename, sep=",")
    eq_df["time"] = pd.to_datetime(eq_df["time"])
    eq_df = eq_df[(eq_df["time"] >= d - pd.Timedelta(days=shift))]
    eq_df.index = range(len(eq_df))
    eq_df = eq_df[(eq_df["time"] <= pd.to_datetime(d_end))]
    eq_df["date"] = eq_df["time"]
    eq_df = eq_df[["date", "mag"]]
    eq_df = bin_origin_start(eq_df, "date", "mag", int(t[:-1]), ["count", "sum"])
    eq_df.at[0, "count"] = eq_df.at[0, "count"] - 1

    d = d - pd.Timedelta(days=shift2)
    d_end = d + pd.Timedelta(days=l * int(shift))

    cr_df = pd.read_csv(THEIR_DIR + cr_filename, sep=",")
    if station == "PA":
        cr_df["date"] = pd.to_datetime(cr_df["time"], unit="s")
    else:
        cr_df["date"] = pd.to_datetime(cr_df["datetime"], format="%Y-%m-%d %H:%M:%S")

    cr_df = cr_df[(cr_df["date"] >= pd.to_datetime(d) - pd.Timedelta(days=shift))]
    cr_df.index = range(len(cr_df))
    cr_df = cr_df[(cr_df["date"] <= pd.to_datetime(d_end))]
    if station == "PA":
        cr_df = bin_origin_start(cr_df, "date", "rateCorr", int(t[:-1]), ["mean", "sum", "count"])
    else:
        cr_df = bin_origin_start(cr_df, "date", "value", int(t[:-1]), ["mean", "sum", "count"])

    c = pd.DataFrame()
    c["cr date"] = cr_df["date"]
    c["eq date"] = eq_df["date"]
    c["cr sum"] = cr_df["sum"]
    c["eq sum"] = eq_df["sum"]
    c["cr mean"] = cr_df["mean"]
    c["cr delta3"] = abs(c["cr mean"].diff())
    c.index = range(len(c))
    c = c[(c["cr mean"] > 0) & (c["cr delta3"] > 0)]
    c["cr median"] = c["cr delta3"].median()
    c["eq median"] = c["eq sum"].median()
    c["A"] = c["eq sum"] / c["eq median"] - 1
    c["B"] = c["cr delta3"] / c["cr median"] - 1
    c["C"] = np.sign(c["A"] * c["B"])
    c.index = range(len(c))

    tmp_c = c[(c["C"] >= 0)]
    CDF = 1 - binom.cdf(len(tmp_c), len(c), 0.5, 1)
    n = c["C"].value_counts().get(1, 0)
    N = len(c)
    PDF = math.comb(N, int(n)) * ((1 / 2) ** N)
    return dict(N=N, Np=int(n), Nm=int(N - n), PDF=PDF, CDF=CDF), c


In [ ]:
# KROK 4: ICH kod, wersja pdf_values6.py (druga, wcześniejsza wersja -
# BEZ ostrzeżenia "NIE DOKONCA" na górze pliku, ale z inną obsługą
# sztucznego wiersza startowego i innym filtrem - patrz różnice opisane
# w rozmowie: tu synthetic new_row JEST dodawany, i brakuje filtra
# "cr mean>0", tylko "cr delta3>0"). Skopiowana 1:1 poza parametryzacją
# ścieżek plików, tak jak wyżej.
def their_pdf_v6(d, l, t, shift, shift2, station, cr_filename, eq_filename="eq_data.csv"):
    d = pd.to_datetime(d) + pd.Timedelta(days=shift2)
    d_end = d + pd.Timedelta(days=l * int(t[:-1]))

    eq_df = pd.read_csv(THEIR_DIR + eq_filename, sep=",")
    eq_df["time"] = pd.to_datetime(eq_df["time"])
    eq_df = eq_df[(eq_df["time"] >= d - pd.Timedelta(days=shift))]
    eq_df.index = range(len(eq_df))
    new_row_eq = pd.DataFrame({"time": pd.to_datetime(d) - pd.Timedelta(days=shift),
                                "latitude": 0, "longitude": 0, "mag": 0}, index=[0])
    eq_df = pd.concat([new_row_eq, eq_df]).reset_index(drop=True)
    eq_df = eq_df[(eq_df["time"] <= pd.to_datetime(d_end))]
    eq_df["date"] = eq_df["time"]
    eq_df = eq_df[["date", "mag"]]
    eq_df = bin_origin_start(eq_df, "date", "mag", int(t[:-1]), ["count", "sum"])
    eq_df.at[0, "count"] = eq_df.at[0, "count"] - 1

    d = d - pd.Timedelta(days=shift2)
    d_end = d + pd.Timedelta(days=l * int(shift))

    cr_df = pd.read_csv(THEIR_DIR + cr_filename, sep=",")
    if station == "PA":
        cr_df["date"] = pd.to_datetime(cr_df["time"], unit="s")
        new_row_cr = pd.DataFrame({"time": 0, "rateCorr": 0, "arrayFraction": 0, "rateUncorr": 0,
                                    "pressure": 0, "date": pd.to_datetime(d) - pd.Timedelta(days=shift)}, index=[0])
    else:
        cr_df["date"] = pd.to_datetime(cr_df["datetime"], format="%Y-%m-%d %H:%M:%S")
        new_row_cr = pd.DataFrame({"time": 0, "value": 0, "date": pd.to_datetime(d) - pd.Timedelta(days=shift)}, index=[0])

    cr_df = cr_df[(cr_df["date"] > pd.to_datetime(d) - pd.Timedelta(days=shift))]
    cr_df.index = range(len(cr_df))
    cr_df = pd.concat([new_row_cr, cr_df]).reset_index(drop=True)
    cr_df = cr_df[(cr_df["date"] <= pd.to_datetime(d_end))]
    if station == "PA":
        cr_df = bin_origin_start(cr_df, "date", "rateCorr", int(t[:-1]), ["mean", "sum", "count"])
    else:
        cr_df = bin_origin_start(cr_df, "date", "value", int(t[:-1]), ["mean", "sum", "count"])
    cr_df.at[0, "count"] = cr_df.at[0, "count"] - 1

    c = pd.DataFrame()
    c["cr date"] = cr_df["date"]
    c["eq date"] = eq_df["date"]
    c["cr sum"] = cr_df["sum"]
    c["eq sum"] = eq_df["sum"]
    c["cr mean"] = cr_df["mean"]
    c["cr delta3"] = abs(c["cr mean"].diff())
    c.index = range(len(c))
    c["cr median"] = c["cr delta3"].median()
    c = c[(c["cr delta3"] > 0)]
    c["eq median"] = c["eq sum"].median()
    c["A"] = c["eq sum"] / c["eq median"] - 1
    c["B"] = c["cr delta3"] / c["cr median"] - 1
    c["c"] = np.sign(c["A"] * c["B"])
    c.index = range(len(c))

    tmp_c = c[(c["c"] > 0)]
    CDF = 1 - binom.cdf(len(tmp_c), len(c), 0.5, 1)
    n = c["c"].value_counts().get(1, 0)
    N = len(c)
    PDF = math.comb(N, int(n)) * ((1 / 2) ** N)
    return dict(N=N, Np=int(n), Nm=int(N - n), PDF=PDF, CDF=CDF), c


In [ ]:
# KROK 5: odpalenie ICH kodu (obu wersji) na ICH danych (mosc_data2.csv -
# ten, który zgadza się z naszym mosc_data.csv, patrz KROK 2 - mosc_data1
# to prawdopodobnie wariant %, pomijamy go tutaj) + ICH eq_data.csv.
# Porównanie z tym, co jest zapisane w ich pliku
# pdf_Mosc_2013-11-14 07-00-00.csv (214/118 - policzone ręcznie w
# rozmowie, tutaj liczymy to jeszcze raz z samego pliku, żeby nie polegać
# na niczym "na słowo").
result_v8, bins_v8 = their_pdf_v8("2013-11-14 07:00:00", 335, "5D", 5, 15, "Mosc",
                                   cr_filename="mosc_data2.csv")
result_v6, bins_v6 = their_pdf_v6("2013-11-14 07:00:00", 335, "5D", 5, 15, "Mosc",
                                   cr_filename="mosc_data2.csv")

saved_csv = pd.read_csv(THEIR_DIR + "pdf_Mosc_2013-11-14 07-00-00.csv")
saved_Np = int((saved_csv["C"] == 1.0).sum())
saved_Nm = int((saved_csv["C"] == -1.0).sum())

print(f"ich kod v8 (pdf_values8.py, na mosc_data2.csv): N={result_v8['N']}, "
      f"Np={result_v8['Np']}, Nm={result_v8['Nm']}, PDF={result_v8['PDF']:.3e}, CDF={result_v8['CDF']:.3e}")
print(f"ich kod v6 (pdf_values6.py, na mosc_data2.csv): N={result_v6['N']}, "
      f"Np={result_v6['Np']}, Nm={result_v6['Nm']}, PDF={result_v6['PDF']:.3e}, CDF={result_v6['CDF']:.3e}")
print(f"ich ZAPISANY plik (pdf_Mosc_2013-11-14...csv, policzone tutaj z surowego pliku): "
      f"N={len(saved_csv)}, Np={saved_Np}, Nm={saved_Nm}")
print(f"ARTYKUŁ (opublikowane): Np={ARTICLE_RESULT['Np']}, Nm={ARTICLE_RESULT['Nm']}")
print(f"NASZ kod (z KROKU 1): Np={baseline_stat['Np']}, Nm={baseline_stat['Nm']}")


In [ ]:
# KROK 6: rozdzielenie "czy to dane czy kod" - nasz kod (cosmoseismic_stat)
# na ICH danych (mosc_data2.csv + eq_data.csv), zamiast naszych własnych
# plików. Jeśli to da coś bliższego 206/128 niż 214/118 - różnica jest w
# KODZIE (sposobie liczenia binów), nie w danych źródłowych.
def load_their_eq(filename="eq_data.csv"):
    df = pd.read_csv(THEIR_DIR + filename, usecols=["time", "mag"])
    df["time"] = pd.to_datetime(df["time"])
    return df.set_index("time")["mag"].sort_index()

eq_theirs = load_their_eq()
cr_theirs_2_full = load_their_cr("mosc_data2.csv")

stat_ourcode_theirdata, _ = cosmoseismic_stat_detailed(
    cr_theirs_2_full, eq_theirs, T0, P_DAYS, D_DAYS, M_THRESHOLD, DT_DAYS)
print(f"NASZ kod + ICH dane (mosc_data2.csv, eq_data.csv): "
      f"N_valid={stat_ourcode_theirdata['N_valid']}, Np={stat_ourcode_theirdata['Np']}, "
      f"Nm={stat_ourcode_theirdata['Nm']}, sigma={stat_ourcode_theirdata['sigma']:.3f}")

stat_ourcode_theireq, _ = cosmoseismic_stat_detailed(
    cr_ours, eq_theirs, T0, P_DAYS, D_DAYS, M_THRESHOLD, DT_DAYS)
print(f"NASZ kod + NASZE CR + ICH EQ (eq_data.csv): "
      f"N_valid={stat_ourcode_theireq['N_valid']}, Np={stat_ourcode_theireq['Np']}, "
      f"Nm={stat_ourcode_theireq['Nm']}, sigma={stat_ourcode_theireq['sigma']:.3f}")


In [ ]:
# KROK 7: porównanie bin-po-binie - nasza tabela (baseline_bins z KROKU 1)
# vs ich zapisany plik (saved_csv z KROKU 5).
#
# POPRAWKA wyrównania (poprzednia wersja porównywała po POZYCJI, co dało
# złudne "51.5% zgodności" - bliskie losowym 50/50, bo:
# 1) nasze biny zaczynają się jeden bin PÓŹNIEJ niż ich (2013-11-19 vs
#    2013-11-14 - patrz wypisane niżej pierwsze daty), więc proste
#    porównanie pozycja-do-pozycji było przesunięte o 1;
# 2) ale samo przesunięcie o 1 TEŻ by nie wystarczyło: nasz filtr "valid"
#    odrzuca inną liczbę wierszy niż ich filtr (u nas N_valid=334/335,
#    u nich N=332/335 - różne wiersze mogły zostać odrzucone w środku
#    szeregu, nie tylko na brzegach), więc stały offset pozycji może się
#    rozjechać w dowolnym miejscu środka szeregu.
# Rozwiązanie: łączymy po DACIE (zaokrąglonej do dnia - biny są odległe o
# 5 dni, więc żadne dwa biny nie wypadają tego samego dnia, to bezpieczny
# klucz), a nie po pozycji - to odporne na różną liczbę odrzuconych
# wierszy po obu stronach.
print("Pierwsze daty (nasze vs ich, przed wyrównaniem - widać przesunięcie o 1 bin):")
print(baseline_bins["cr_bin_start"].head(3).to_list())
print(pd.to_datetime(saved_csv["cr date"]).head(3).to_list())

baseline_bins["day"] = baseline_bins["cr_bin_start"].dt.floor("D")
saved_csv["day"] = pd.to_datetime(saved_csv["cr date"]).dt.floor("D")

nasz_c = pd.DataFrame({
    "day": baseline_bins["day"],
    "nasz_cr_date": baseline_bins["cr_bin_start"],
    "nasz_C": np.sign(baseline_bins["A"] * baseline_bins["B"]),
})
ich_c = pd.DataFrame({
    "day": saved_csv["day"],
    "ich_cr_date": pd.to_datetime(saved_csv["cr date"]),
    "ich_C": saved_csv["C"],
})

cmp_df = pd.merge(nasz_c, ich_c, on="day", how="inner")
cmp_df["zgodny_znak"] = cmp_df["nasz_C"] == cmp_df["ich_C"]
print(f"\nWspólnych dni (po wyrównaniu po dacie): {len(cmp_df)} "
      f"(nasze: {len(baseline_bins)}, ich: {len(saved_csv)})")
print(f"zgodny znak w: {cmp_df['zgodny_znak'].sum()} ({cmp_df['zgodny_znak'].mean()*100:.1f}%), "
      f"NIEZGODNY w: {(~cmp_df['zgodny_znak']).sum()}")
cmp_df[~cmp_df["zgodny_znak"]][["nasz_cr_date", "ich_cr_date", "nasz_C", "ich_C"]].head(20)


In [ ]:
# KROK 8 (bonus, szybki): to samo poglądowo dla Oulu - z zapisanego pliku
# pdf_Oulu_2014-01-04 23-37-12.csv, żeby sprawdzić czy wzorzec
# "ich zapisany plik != opublikowane" powtarza się też tam.
saved_oulu = pd.read_csv(THEIR_DIR + "pdf_Oulu_2014-01-04 23-37-12.csv")
oulu_Np = int((saved_oulu["C"] == 1.0).sum())
oulu_Nm = int((saved_oulu["C"] == -1.0).sum())
print(f"Oulu, ich zapisany plik: N={len(saved_oulu)}, Np={oulu_Np}, Nm={oulu_Nm}")
print("Oulu, ARTYKUŁ (opublikowane): Np=220, Nm=112")


In [ ]:
# KROK 9: próba faktycznej replikacji - ICH sposób binowania (bin_origin_start,
# odtworzone origin="start") + PEŁNY filtr z równania (3) artykułu
# (Ai!=0, Bi!=0, nCR(ti)>0, nCR(ti-1)>0, Sm(ti+dt)>0), zamiast okrojonego
# filtra z pdf_values6.py/pdf_values8.py (tylko cr_mean>0 & cr_delta3>0 -
# BRAK sprawdzenia nCR(ti-1)>0 i Sm>0, mimo że artykuł wymaga tego wprost).
#
# Uzasadnienie tej hipotezy: liczba ważnych binów N maleje w kolejności
# NASZ KOD (334) -> ICH KOD zapisany/odtworzony (332) -> PUBLIKACJA (331) -
# to sugeruje, że publikacja używała NAJBARDZIEJ restrykcyjnego filtra,
# ściślej trzymającego się równania (3) niż to, co przetrwało w archiwum
# jako pdf_values6/8.py. To najlepiej uzasadniona, dotychczas
# NIEPRZETESTOWANA hipoteza z całej tej analizy - test poniżej.
def hybrid_pdf(d, l, t, shift, shift2, cr_path, eq_path):
    d = pd.to_datetime(d) + pd.Timedelta(days=shift2)
    d_end = d + pd.Timedelta(days=l * int(t[:-1]))

    eq_df = pd.read_csv(eq_path, sep=",")
    eq_df["time"] = pd.to_datetime(eq_df["time"])
    eq_df = eq_df[(eq_df["time"] >= d - pd.Timedelta(days=shift))]
    eq_df.index = range(len(eq_df))
    eq_df = eq_df[(eq_df["time"] <= pd.to_datetime(d_end))]
    eq_df["date"] = eq_df["time"]
    eq_df = eq_df[["date", "mag"]]
    eq_df = bin_origin_start(eq_df, "date", "mag", int(t[:-1]), ["count", "sum"])

    d2 = d - pd.Timedelta(days=shift2)
    d2_end = d2 + pd.Timedelta(days=l * int(shift))

    cr_df = pd.read_csv(cr_path, sep=",")
    cr_df["date"] = pd.to_datetime(cr_df["datetime"], format="%Y-%m-%d %H:%M:%S")
    cr_df = cr_df[(cr_df["date"] >= pd.to_datetime(d2) - pd.Timedelta(days=shift))]
    cr_df.index = range(len(cr_df))
    cr_df = cr_df[(cr_df["date"] <= pd.to_datetime(d2_end))]
    cr_df = bin_origin_start(cr_df, "date", "value", int(t[:-1]), ["mean", "sum", "count"])

    c = pd.DataFrame()
    c["cr date"] = cr_df["date"]
    c["eq date"] = eq_df["date"]
    c["eq sum"] = eq_df["sum"]
    c["cr mean"] = cr_df["mean"]
    c["cr mean prev"] = c["cr mean"].shift(1)
    c["cr delta"] = (c["cr mean"] - c["cr mean prev"]).abs()

    med_Sm = c["eq sum"].median()
    med_dCR = c["cr delta"].median()
    c["A"] = c["eq sum"] / med_Sm - 1
    c["B"] = c["cr delta"] / med_dCR - 1

    valid = (
        (c["A"] != 0) & (c["B"] != 0) &
        (c["cr mean"] > 0) & (c["cr mean prev"] > 0) &
        (c["eq sum"] > 0) &
        c["A"].notna() & c["B"].notna()
    )
    c = c[valid].reset_index(drop=True)
    c["C"] = np.sign(c["A"] * c["B"])

    N = len(c)
    Np = int((c["C"] > 0).sum())
    Nm = int((c["C"] < 0).sum())
    PDF = binom.pmf(Np, N, 0.5)
    CDF = binom.sf(Np - 1, N, 0.5)
    sigma = norm.isf(CDF)
    return dict(N=N, Np=Np, Nm=Nm, PDF=PDF, CDF=CDF, sigma=sigma), c


hybrid_theirs, _ = hybrid_pdf("2013-11-14 07:00:00", 335, "5D", 5, 15,
                               cr_path=THEIR_DIR + "mosc_data2.csv", eq_path=THEIR_DIR + "eq_data.csv")
hybrid_ourcr, _ = hybrid_pdf("2013-11-14 07:00:00", 335, "5D", 5, 15,
                              cr_path=MOSC_PATH, eq_path=THEIR_DIR + "eq_data.csv")

print(f"ich binowanie + PEŁNY filtr + ich CR (mosc_data2): N={hybrid_theirs['N']}, "
      f"Np={hybrid_theirs['Np']}, Nm={hybrid_theirs['Nm']}, sigma={hybrid_theirs['sigma']:.3f}")
print(f"ich binowanie + PEŁNY filtr + NASZ CR (mosc_data.csv): N={hybrid_ourcr['N']}, "
      f"Np={hybrid_ourcr['Np']}, Nm={hybrid_ourcr['Nm']}, sigma={hybrid_ourcr['sigma']:.3f}")
print(f"ARTYKUŁ (cel): Np={ARTICLE_RESULT['Np']}, Nm={ARTICLE_RESULT['Nm']}, N=331")


In [ ]:
# PODSUMOWANIE - wszystkie warianty obok siebie, dla Moskwy, t0=14 lis 2013.
# Uporządkowane w kolejności rosnącej "bliskości" do naszego baseline ->
# artykułu, żeby widzieć postęp tej sesji krok po kroku.
summary = pd.DataFrame([
    dict(wariant="NASZ kod + NASZE dane (baseline, 20260717.txt)", Np=baseline_stat["Np"], Nm=baseline_stat["Nm"]),
    dict(wariant="NASZ kod + NASZE CR + ICH EQ (eq_data.csv)", Np=stat_ourcode_theireq["Np"], Nm=stat_ourcode_theireq["Nm"]),
    dict(wariant="NASZ kod + ICH CR i EQ (mosc_data2, eq_data)", Np=stat_ourcode_theirdata["Np"], Nm=stat_ourcode_theirdata["Nm"]),
    dict(wariant="ich kod pdf_values8.py + ich dane (mosc_data2, eq_data)", Np=result_v8["Np"], Nm=result_v8["Nm"]),
    dict(wariant="ich kod pdf_values6.py + ich dane (mosc_data2, eq_data)", Np=result_v6["Np"], Nm=result_v6["Nm"]),
    dict(wariant="ich zapisany plik pdf_Mosc_...csv (przeliczone tutaj)", Np=saved_Np, Nm=saved_Nm),
    dict(wariant="ich binowanie + PEŁNY filtr (3) + ich CR (mosc_data2)", Np=hybrid_theirs["Np"], Nm=hybrid_theirs["Nm"]),
    dict(wariant="ich binowanie + PEŁNY filtr (3) + NASZ CR (mosc_data.csv)", Np=hybrid_ourcr["Np"], Nm=hybrid_ourcr["Nm"]),
    dict(wariant="ARTYKUŁ (opublikowane, CEL)", Np=ARTICLE_RESULT["Np"], Nm=ARTICLE_RESULT["Nm"]),
])
summary["N"] = summary["Np"] + summary["Nm"]
summary["dystans_do_artykulu (|dNp|+|dNm|)"] = (
    (summary["Np"] - ARTICLE_RESULT["Np"]).abs() + (summary["Nm"] - ARTICLE_RESULT["Nm"]).abs())
print(summary.to_string(index=False))
